# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 6 — Arquitectura RAG

ITBA Educación Ejecutiva

Construcción de un sistema de generación sustentada: hoy conectamos el buscador semántico de la Clase 5 con el modelo generativo de la Clase 3. Esto **es** RAG — todo lo anterior era preparación.

### Cómo se usa este notebook

- El docente corre y explica cada celda en vivo. No programás en simultáneo durante la clase.
- Celdas **🏠 Practicá**: tarea individual para después de la clase.
- **Sección 10 (Laboratorio)**: checkpoint del proyecto integrador.

## Objetivo de la clase

Al final de la clase vas a poder:
- Explicar el flujo consulta → recuperación → contexto → generación, y por qué conviene construirlo explícito antes de usar un framework.
- Diseñar un prompt que delimite instrucciones, evidencia y consulta.
- Gestionar un presupuesto de tokens para el contexto recuperado.
- Lograr que el modelo responda "no lo sé" cuando no hay evidencia suficiente, en vez de inventar.
- Implementar citas que trazan cada afirmación hasta su fragmento de origen.

**Checkpoint del proyecto integrador de hoy**: pipeline RAG end-to-end funcionando (ver `proyecto-integrador/README.md`, checkpoint Clase 6).

## Agenda

1. Arquitectura RAG y el flujo de 4 pasos
2. Por qué construirlo explícito (sin frameworks de alto nivel)
3. Implementación base del pipeline
4. Diseño del prompt con evidencia recuperada
5. Gestión de la ventana de contexto
6. Respuestas sustentadas y manejo de ausencia de evidencia
7. Implementación de citas
8. Registro de documentos y fragmentos utilizados
9. Pruebas sobre el dataset del proyecto
10. Laboratorio: pipeline RAG integrado

## 0. Punto de partida

Recordá de clases anteriores: `embed_bedrock` y `llamar_bedrock` (clientes de Bedrock), y el buscador semántico con ChromaDB (Clase 5). Reconstruimos la versión mínima de cada uno para trabajar hoy.

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6")
EMBED_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_EMBED", "cohere.embed-v4:0")

def embed_bedrock(textos: list[str], input_type: str = "search_document") -> list[list[float]]:
    body = json.dumps({"texts": textos, "input_type": input_type, "embedding_types": ["float"]})
    resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body, accept="*/*", contentType="application/json")
    return json.loads(resp["body"].read())["embeddings"]["float"]

def llamar_bedrock(prompt, system=None, temperature=0.0, max_tokens=500):
    kwargs = {"modelId": MODEL_ID, "messages": [{"role": "user", "content": [{"text": prompt}]}],
              "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens}}
    if system:
        kwargs["system"] = [{"text": system}]
    resp = bedrock.converse(**kwargs)
    return resp["output"]["message"]["content"][0]["text"]

In [ ]:
# Mismo corpus de juguete de la Clase 5, reconstruido en ChromaDB
import chromadb

corpus = [
    {"id": "apis-001", "texto": "La Converse API de Bedrock unifica la forma de llamar a distintos modelos: cambia el modelId, no el código.", "doc": "guia-apis"},
    {"id": "apis-002", "texto": "Los parámetros de inferencia como temperature y maxTokens van en inferenceConfig en Bedrock.", "doc": "guia-apis"},
    {"id": "seguridad-001", "texto": "La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.", "doc": "guia-seguridad"},
    {"id": "seguridad-002", "texto": "El principio de mínimo privilegio limita qué herramientas puede invocar un agente según el contexto.", "doc": "guia-seguridad"},
    {"id": "rag-001", "texto": "Un pipeline RAG combina recuperación de evidencia con generación sustentada en esa evidencia.", "doc": "guia-rag"},
    {"id": "rag-002", "texto": "El top-k determina cuántos fragmentos recuperados se incluyen como contexto para el modelo generador.", "doc": "guia-rag"},
]

cliente_chroma = chromadb.Client()  # en memoria, alcanza para la demo de hoy
coleccion = cliente_chroma.get_or_create_collection("corpus_clase6")
coleccion.add(
    ids=[c["id"] for c in corpus],
    embeddings=embed_bedrock([c["texto"] for c in corpus]),
    documents=[c["texto"] for c in corpus],
    metadatas=[{"doc": c["doc"]} for c in corpus],
)
print("Chunks indexados:", coleccion.count())

## 1. Arquitectura RAG y el flujo de 4 pasos

```
consulta del usuario
       │
       ▼
 [1] RECUPERACIÓN  → embeber la consulta, buscar top-k en el vector store
       │
       ▼
 [2] CONTEXTO       → armar el bloque de evidencia a partir de los chunks recuperados
       │
       ▼
 [3] PROMPT          → combinar instrucciones + evidencia + consulta, delimitados
       │
       ▼
 [4] GENERACIÓN      → el modelo responde usando SOLO esa evidencia
```

Cada paso de este flujo ya lo construiste por separado: recuperación es la Clase 5, generación es la Clase 2-3. Hoy es pura integración — y los detalles de esa integración (cómo armás el contexto, cómo delimitás el prompt) son la mitad del trabajo real de un sistema RAG en producción.

## 2. Por qué construirlo explícito (sin frameworks de alto nivel)

Frameworks como LangChain o LlamaIndex tienen abstracciones para "armar un RAG en 10 líneas" — y una vez que entendés qué hay debajo, pueden ahorrar tiempo. Pero aprenderlos *antes* de entender el flujo tiene un costo: cuando algo no funciona bien (y en RAG, algo siempre anda raro al principio — Clase 7), depurar una abstracción que no entendés es mucho más difícil que depurar tu propio código.

Por eso en este curso construimos el pipeline a mano primero. Si más adelante en tu trabajo usás un framework, vas a saber exactamente qué está pasando detrás de cada función que llamás.

## 3. Implementación base del pipeline

In [ ]:
def recuperar(consulta: str, k: int = 3) -> list[dict]:
    vec = embed_bedrock([consulta], input_type="search_query")[0]
    resultado = coleccion.query(query_embeddings=[vec], n_results=k)
    return [
        {"chunk_id": id_, "texto": texto, "doc": meta["doc"], "distancia": dist}
        for id_, texto, meta, dist in zip(
            resultado["ids"][0], resultado["documents"][0], resultado["metadatas"][0], resultado["distances"][0]
        )
    ]


def armar_contexto(chunks: list[dict]) -> str:
    bloques = [f"[{i+1}] (fuente: {c['doc']}) {c['texto']}" for i, c in enumerate(chunks)]
    return "\n\n".join(bloques)


def rag_v1(consulta: str, k: int = 3) -> str:
    chunks = recuperar(consulta, k)
    contexto = armar_contexto(chunks)
    prompt = f"Contexto:\n{contexto}\n\nPregunta: {consulta}\n\nRespondé usando el contexto."
    return llamar_bedrock(prompt)


print(rag_v1("¿Qué controla la temperatura en una llamada al modelo?"))

Esto ya "funciona", pero es frágil: no delimita claramente contexto/pregunta, no define qué hacer si no hay evidencia, y no dice de dónde sacó la respuesta. Las próximas 4 secciones arreglan cada uno de esos problemas, uno a la vez.

## 4. Diseño del prompt con evidencia recuperada

Igual que en la Clase 2 (delimitación de instrucciones y contexto), separar claramente **instrucciones**, **evidencia** y **consulta** reduce ambigüedad — y es la primera defensa contra que el modelo confunda una instrucción con un dato del contexto (prompt injection indirecta vía documentos recuperados, Clase 11).

In [ ]:
SYSTEM_RAG = (
    "Sos un asistente que responde preguntas usando EXCLUSIVAMENTE la evidencia provista. "
    "No uses conocimiento propio ni inventes información que no esté en la evidencia."
)

def armar_prompt_delimitado(consulta: str, contexto: str) -> str:
    return f'''<evidencia>
{contexto}
</evidencia>

<pregunta>
{consulta}
</pregunta>

Respondé la pregunta usando solo la evidencia. Si la evidencia no alcanza, decilo explícitamente.'''


def rag_v2(consulta: str, k: int = 3) -> str:
    chunks = recuperar(consulta, k)
    contexto = armar_contexto(chunks)
    prompt = armar_prompt_delimitado(consulta, contexto)
    return llamar_bedrock(prompt, system=SYSTEM_RAG)


print(rag_v2("¿Qué controla la temperatura en una llamada al modelo?"))

**🏠 Practicá**: compará `rag_v1` y `rag_v2` con la misma consulta pero pasándole, a propósito, un `contexto` con información contradictoria (ej. agregá un chunk falso que diga "la temperatura se configura en `generationConfig`, no en `inferenceConfig`") — fijate si alguna de las dos versiones es más resistente a ese tipo de contaminación del contexto.

## 5. Gestión de la ventana de contexto

El contexto recuperado no es gratis: cada chunk que agregás consume tokens de la ventana (Clase 1) y de tu presupuesto de costo (Clase 3). Con `k` fijo, un corpus con chunks largos puede desbordar silenciosamente el presupuesto de la llamada.

In [ ]:
def armar_contexto_con_presupuesto(chunks: list[dict], max_caracteres: int = 800) -> str:
    """Agrega chunks al contexto hasta agotar el presupuesto de caracteres (proxy simple de tokens)."""
    bloques, total = [], 0
    for i, c in enumerate(chunks):
        bloque = f"[{i+1}] (fuente: {c['doc']}) {c['texto']}"
        if total + len(bloque) > max_caracteres:
            print(f"Presupuesto agotado: se descartan {len(chunks) - i} chunk(s) de menor ranking.")
            break
        bloques.append(bloque)
        total += len(bloque)
    return "\n\n".join(bloques)


chunks_prueba = recuperar("¿Qué controla la temperatura?", k=6)  # pedimos todos los que hay
contexto_acotado = armar_contexto_con_presupuesto(chunks_prueba, max_caracteres=150)
print(contexto_acotado)

En producción, este presupuesto se calcula con un tokenizador real (no caracteres) y se resta de la ventana de contexto total del modelo menos lo que reservás para la respuesta — un detalle que en un corpus con chunks largos o modelos con ventana chica puede ser la diferencia entre que el sistema funcione o tire un error de contexto excedido.

**🏠 Practicá**: ¿qué pasaría si `max_caracteres` fuera menor al tamaño de un solo chunk? Modificá la función para que, en ese caso extremo, trunque el chunk en vez de descartarlo entero.

## 6. Respuestas sustentadas y manejo de ausencia de evidencia

El problema más común de un RAG mal instruido: cuando la evidencia no alcanza, el modelo **de todas formas responde** usando su conocimiento general — pareciendo sustentado cuando no lo está. Hay que instruirlo explícitamente a reconocer esa situación.

In [ ]:
consulta_sin_evidencia = "¿Cómo se hace un backup de la base de datos?"  # no está en el corpus, a propósito

print("--- rag_v1 (sin instrucción de abstenerse) ---")
print(rag_v1(consulta_sin_evidencia))

print("\n--- rag_v2 (con instrucción explícita) ---")
print(rag_v2(consulta_sin_evidencia))

Si `rag_v2` todavía inventa una respuesta, el problema casi siempre es que el top-k devolvió *algo* (Clase 5: la similitud nunca es cero) y el modelo lo tomó como evidencia válida aunque sea irrelevante. La solución combina dos cosas que ya tenés: el **umbral de similitud** de la Clase 5 (no pasar chunks de baja relevancia al contexto) y una instrucción de sistema que explícitamente autorice a decir "no lo sé".

In [ ]:
def rag_v3(consulta: str, k: int = 3, umbral_distancia: float = 1.0) -> str:
    chunks = recuperar(consulta, k)
    chunks_relevantes = [c for c in chunks if c["distancia"] < umbral_distancia]  # Chroma: menor distancia = más relevante

    if not chunks_relevantes:
        return "No encontré evidencia suficiente en el corpus para responder esta pregunta."

    contexto = armar_contexto(chunks_relevantes)
    prompt = armar_prompt_delimitado(consulta, contexto)
    return llamar_bedrock(prompt, system=SYSTEM_RAG)


print(rag_v3(consulta_sin_evidencia, umbral_distancia=0.9))
print()
print(rag_v3("¿Qué controla la temperatura en una llamada al modelo?"))

**🏠 Practicá**: el umbral `0.9` de arriba lo elegí mirando los datos de este corpus de juguete — no es un valor universal. Corré `recuperar(consulta_sin_evidencia, k=6)` y mirá las distancias reales de los 6 chunks para calibrar un umbral que tenga sentido para *este* corpus específico.

## 7. Implementación de citas

Una respuesta sustentada debería poder señalar *de dónde* sacó cada afirmación — no solo confiar en que el modelo "usó" la evidencia, sino poder trazarlo.

In [ ]:
def rag_con_citas(consulta: str, k: int = 3, umbral_distancia: float = 1.0) -> dict:
    chunks = recuperar(consulta, k)
    chunks_relevantes = [c for c in chunks if c["distancia"] < umbral_distancia]

    if not chunks_relevantes:
        return {"respuesta": "No encontré evidencia suficiente para responder.", "fuentes": []}

    contexto = armar_contexto(chunks_relevantes)
    instruccion_citas = (
        "Además, citá la evidencia usada con el número entre corchetes correspondiente, "
        "ej. 'La temperatura se configura en inferenceConfig [1].'"
    )
    prompt = armar_prompt_delimitado(consulta, contexto) + "\n\n" + instruccion_citas
    respuesta = llamar_bedrock(prompt, system=SYSTEM_RAG)

    fuentes = [{"ref": i + 1, "chunk_id": c["chunk_id"], "doc": c["doc"]} for i, c in enumerate(chunks_relevantes)]
    return {"respuesta": respuesta, "fuentes": fuentes}


resultado = rag_con_citas("¿Qué controla la temperatura en una llamada al modelo?")
print(resultado["respuesta"])
print("\nFuentes citadas (todas las disponibles en el contexto, el modelo elige cuáles usar):")
for f in resultado["fuentes"]:
    print(f"  [{f['ref']}] {f['chunk_id']} ({f['doc']})")

Esto es "citas por convención del prompt" — el modelo puede, en teoría, citar mal o no citar. Para citas *garantizadas* por el sistema (no por buena voluntad del modelo) hace falta post-procesar la respuesta y validar que cada `[n]` citado exista en las fuentes — queda para profundizar en la Clase 7 (debugging de RAG).

**🏠 Practicá**: agregá una validación simple con regex que extraiga todos los `[n]` de la respuesta y confirme que cada uno corresponde a un `ref` real de `fuentes` — si el modelo citó `[4]` y solo hay 3 fuentes, es una señal de alucinación de cita.

## 8. Registro de documentos y fragmentos utilizados

Más allá de las citas que ve el usuario final, para debugging (Clase 7) y evaluación (Clase 8) conviene loguear internamente *qué se recuperó*, no solo qué se citó — a veces el problema no es la generación, es que la recuperación trajo lo que no correspondía.

In [ ]:
registro_consultas = []  # en producción: log estructurado o base de datos, no una lista en memoria

def rag_con_registro(consulta: str, k: int = 3, umbral_distancia: float = 1.0) -> dict:
    chunks = recuperar(consulta, k)
    resultado = rag_con_citas(consulta, k, umbral_distancia)

    registro_consultas.append({
        "consulta": consulta,
        "chunks_recuperados": [c["chunk_id"] for c in chunks],
        "distancias": [round(c["distancia"], 3) for c in chunks],
        "chunks_usados_en_contexto": [f["chunk_id"] for f in resultado["fuentes"]],
    })
    return resultado


rag_con_registro("¿Qué es la inyección de prompts?")
print(registro_consultas[-1])

Notá la diferencia entre `chunks_recuperados` (todo lo que trajo el top-k) y `chunks_usados_en_contexto` (lo que sobrevivió el filtro de umbral) — cuando algo falla, esta distinción te dice si el problema fue de recuperación (top-k traía basura) o de umbral/generación (la evidencia correcta estaba ahí pero se descartó o no se usó bien).

## 9. Pruebas sobre el dataset del proyecto

Antes del laboratorio, una ronda rápida de pruebas manuales con preguntas variadas — relevantes, ambiguas, y fuera de dominio — sobre el corpus de juguete de hoy.

In [ ]:
preguntas_de_prueba = [
    "¿Qué controla la temperatura en una llamada al modelo?",
    "¿Qué es el principio de mínimo privilegio?",
    "¿Para qué sirve el top-k en RAG?",
    "¿Cómo se hace un backup de la base de datos?",  # fuera de dominio
]

for q in preguntas_de_prueba:
    r = rag_con_registro(q)
    print(f"Q: {q}\nA: {r['respuesta']}\n")

**🏠 Practicá**: para cada respuesta de arriba, juzgá si está realmente sustentada en las fuentes citadas (no solo si "suena bien") — es el mismo ejercicio de evaluación manual de la Clase 5, ahora aplicado al pipeline completo.

## 10. Laboratorio: pipeline RAG integrado

**🏠 Tarea de la semana — entregar/completar antes de la Clase 7.**

**Checkpoint del proyecto integrador (Clase 6)**: pipeline RAG end-to-end (consulta → recuperación → contexto → respuesta generada) funcionando sobre el corpus real del TP.

### Consigna

1. Conectá tu buscador semántico real de la Clase 5 (no el corpus de juguete de hoy) con tu `LLMProvider` de la Clase 3.
2. Completá la función `rag` de abajo integrando: prompt delimitado (sección 4), presupuesto de contexto (sección 5), umbral de relevancia + mensaje de "no sé" (sección 6), y citas con registro de fuentes (secciones 7-8).
3. Corré al menos 4 preguntas reales sobre tu corpus: 2 que deberían tener buena respuesta, 1 ambigua, 1 claramente fuera de dominio.
4. Revisá el `registro_consultas` resultante y confirmá que podés explicar, para cada pregunta, por qué el sistema respondió lo que respondió (qué se recuperó, qué pasó el umbral, qué se citó).

### Criterio de éxito

- La pregunta fuera de dominio devuelve el mensaje de "no encontré evidencia", no una respuesta inventada.
- Cada respuesta con evidencia incluye al menos una cita que corresponde a un chunk real y verificable del corpus.
- El pipeline no desborda el presupuesto de contexto definido, incluso si el top-k trae chunks largos.

In [ ]:
def rag(consulta: str, k: int = 3, umbral_distancia: float = 1.0, max_caracteres_contexto: int = 1500) -> dict:
    # TODO: recuperar del vector store real del TP (no `coleccion` de juguete)
    # TODO: filtrar por umbral_distancia
    # TODO: armar contexto respetando max_caracteres_contexto (sección 5)
    # TODO: armar prompt delimitado + instrucción de citas (secciones 4 y 7)
    # TODO: llamar al LLMProvider real (Clase 3), no llamar_bedrock directo si tu proveedor es otro
    # TODO: registrar la consulta (sección 8) y devolver {"respuesta":, "fuentes":}
    raise NotImplementedError


# Prueba (va a fallar hasta que completes el TODO):
# print(rag("una pregunta real sobre tu corpus"))

## Cierre

- Hoy nació RAG propiamente dicho: recuperación + generación, con prompt delimitado, presupuesto de contexto, manejo de ausencia de evidencia y citas trazables.
- La versión de hoy funciona, pero es la **versión base** — en la práctica van a aparecer casos donde falla (recupera mal, cita mal, o se desborda). Eso es exactamente el tema de la próxima clase.
- Próxima clase: RAG avanzado y debugging — diagnóstico sistemático de cada etapa del pipeline y técnicas para mejorarlo (query rewriting, multi-query, context compression).
- Antes de la próxima clase: dejar el laboratorio de la sección 10 funcionando sobre el corpus real del TP.